## 0. Imports

In [1]:
import json
import re
from datetime import date

import os
from pathlib import Path
from dotenv import load_dotenv


import numpy as np
import pandas as pd

# v3 standard mapping helpers (docs/reference/ is the source of truth)
from mold_v3_mapping import (
    MOLD_TYPES,
    POSITION_NAMES,
    STAGES,
    ASSET_STATUSES,
    MOLD_SETS,
    SOLE_TYPE_TO_TYPE,
    normalize_family,
    is_legacy_family_code,
    derive_component,
    build_mold_id,
    component_description,
    derive_construction_type,
)

## 1. Load Raw Data

In [2]:
load_dotenv()

root = Path(os.getenv("DATA_ROOT"))
mold_list       = root/ "WWW Bottom Mold list- All brands - 2026 (2).xlsx"
management_file = root/ "Mold Capacity Management.xlsm"

mold_list_raw_df             = pd.read_excel(mold_list,       sheet_name="All Brands -done")
vendor_master_raw_df         = pd.read_excel(mold_list,       sheet_name="Vendor Master")
mold_style_raw_df            = pd.read_excel(management_file, sheet_name="Style Mold Info")
mold_vendor_allcation_raw_df = pd.read_excel(management_file, sheet_name="Mold Vendor Allocation")
special_mold_raw_df          = pd.read_excel(management_file, sheet_name="Special Molds")

D:\Project\osms-mold-master-data\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


## 2. Helper Functions

In [3]:
# --- DataFrame cleaning ---

def clean_df_col_names(df):
    cols = {}
    for col in df.columns:
        new_col = str(col).split("\n")[0].strip()
        cols[col] = new_col.strip().lower().replace(' ', '_').replace('-', '_')
    df.rename(columns=cols, inplace=True)
    return df


def replace_whitespace(df):
    return df.replace(
        to_replace=[
            r"^\s*$",   # empty or whitespace-only
            r"^NA$",     # NA
            r"^N/A$",    # N/A
            r"^\xa0$",  # non-breaking space
        ],
        value=np.nan,
        regex=True,
    )


def normalize_col_name(col_name):
    return str(col_name).split("\n")[0].strip().lower().replace(" ", "_").replace("-", "_")


def pick_col(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None


# --- Value normalisers ---

def none_if_nan(x):
    if pd.isna(x):
        return None
    return x


def to_int_or_none(x):
    x = none_if_nan(x)
    if x is None:
        return None
    try:
        return int(float(x))
    except Exception:
        return x


def to_float_or_none(x):
    x = none_if_nan(x)
    if x is None:
        return None
    try:
        return float(x)
    except Exception:
        return x


def normalize_numeric_or_text(x):
    x = none_if_nan(x)
    if x is None:
        return None
    try:
        f = float(x)
        return int(f) if f.is_integer() else f
    except Exception:
        return str(x).strip()


def normalize_code(s):
    s = none_if_nan(s)
    if s is None:
        return None
    s = str(s).strip().upper()
    s = re.sub(r"[^A-Z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or None


def normalize_mold_code(s):
    s = none_if_nan(s)
    if s is None:
        return None
    return str(s).strip().upper()


def normalize_season(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip().upper()
    replacements = {"SPR": "S", "SS": "S", "AW": "F", "FW": "F"}
    for old, new in replacements.items():
        if value.startswith(old):
            value = value.replace(old, new, 1)
    return value if re.match(r'^[SF]\d{2}$', value) else np.nan


def normalize_location(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip().title()
    aliases = {
        "North Vietnam": "Vietnam",
        "South Vietnam": "Vietnam",
        "Viet Nam": "Vietnam",
        "Indonesia ": "Indonesia",
    }
    return aliases.get(value, value)


def normalize_location_code(location_name):
    location_name = none_if_nan(location_name)
    if location_name is None:
        return None
    key = str(location_name).strip().title()
    location_code_map = {
        "Vietnam": "VN", "Indonesia": "ID", "India": "IN", "China": "CN",
        "Cambodia": "KH", "Thailand": "TH", "Myanmar": "MM", "Bangladesh": "BD",
        "Pakistan": "PK", "Philippines": "PH", "Taiwan": "TW", "Korea": "KR",
        "South Korea": "KR", "Japan": "JP", "Usa": "US", "United States": "US",
        "Mexico": "MX",
    }
    if key in location_code_map:
        return location_code_map[key]
    fallback = re.sub(r"[^A-Za-z]", "", key).upper()
    return fallback[:2] if fallback else None

LOCATION_CODE_TO_NAME = {
    "VN": "Vietnam",    "CN": "China",         "ID": "Indonesia",
    "BD": "Bangladesh", "KH": "Cambodia",      "TH": "Thailand",
    "MM": "Myanmar",    "IN": "India",          "PK": "Pakistan",
    "PH": "Philippines","TW": "Taiwan",         "KR": "South Korea",
    "JP": "Japan",      "US": "United States",  "MX": "Mexico",
}


def component_code_from_name(component_name):
    s = none_if_nan(component_name)
    if s is None:
        return None
    s = str(s).strip()
    m = re.match(r"^([^\s(]+)", s)
    return m.group(1) if m else s


def ensure_list(x):
    x = none_if_nan(x)
    if x is None:
        return []
    if isinstance(x, (list, tuple, set)):
        return list(x)
    return [str(x)]


# Size columns: "1", "1.5", ..., "18"
size_cols = [str(x).replace(".0", "") for x in np.arange(1, 18.5, 0.5)]

## 3. Clean & Transform DataFrames

### 3a. Mold List -> `df`

In [ ]:
df = mold_list_raw_df.copy()
df = clean_df_col_names(df)
df = replace_whitespace(df)

# Cast columns to target dtypes
dtype_set = {
    'brand': str, 'part_name': str, 'mold_code': str, 'compound': str,
    'vendor_name': str, 'location': str, 'mold_shop': str, 'season': str,
    'a_mold_cost': float, 'last_demand_season': str, 'mold_ownership': str,
    'mold_condition': str, 'daily_output': float, 'total_mold_qty': float,
    '1': float, '1.5': float, '2': float, '2.5': float, '3': float, '3.5': float,
    '4': float, '4.5': float, '5': float, '5.5': float, '6': float, '6.5': float,
    '7': float, '7.5': float, '8': float, '8.5': float, '9': float, '9.5': float,
    '10': float, '10.5': float, '11': float, '11.5': float, '12': float,
    '17.5': float, '18': float,
    'comments': str, 'remark': str, 'actual_output': str,
}
for col, dtype in dtype_set.items():
    if col in df.columns and dtype == str:
        df[col] = df[col].astype(dtype)
    elif col in df.columns and dtype == float:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    else:
        print(f"Warning: Column '{col}' not found in DataFrame.")

# Normalize seasons, brand list, vendor name, location
df['season']             = df['season'].apply(normalize_season)
df['last_demand_season'] = df['last_demand_season'].apply(normalize_season)
df['brand']              = df['brand'].apply(lambda x: x.split("/"))
df['vendor_name']        = df['vendor_name'].str.strip().str.upper()
df['location']           = df['location'].apply(normalize_location)

# Keep only Saucony Outsole / Midsole rows
df = df[(df['part_name'].isin(['Outsole', 'Midsole', 'Other'])) & (df['brand'].apply(lambda brands: 'Saucony' in brands))]


### 3b. Style-Mold mapping -> `mold_style_df`

In [5]:
mold_style_df = mold_style_raw_df.iloc[:, :4].copy()
mold_style_df.columns = ['Brand', 'Style_Name', 'Outsole_Name', 'Midsole_Name']
mold_style_df = replace_whitespace(mold_style_df)
mold_style_df = mold_style_df.dropna(subset=['Brand', 'Style_Name'], how='all')
# All rows are kept — the mismatch case (OS family ≠ MS family) is now handled
# correctly in the style map builder (section 4b) via soleTypes.

### 3c. Vendor allocation -> `mold_vendor_allocation_df`

In [6]:
mold_vendor_allocation_df = mold_vendor_allcation_raw_df.copy()
mold_vendor_allocation_df.columns = mold_vendor_allocation_df.iloc[0]
mold_vendor_allocation_df = mold_vendor_allocation_df[1:]
# Sole Type column dropped — component is identified by Sole Part alone in v3
mold_vendor_allocation_df = mold_vendor_allocation_df[
    ['Factory Number', 'Mold Code', 'Sole Part', 'Vendor ID']
]

## 4. Build Lookup Maps

### 4a. `vendor_reference_map`  (keyed by Vendor ID)

In [7]:
vendor_master_df = vendor_master_raw_df.copy()
vendor_master_df.columns = [normalize_col_name(c) for c in vendor_master_df.columns]
vendor_master_df = replace_whitespace(vendor_master_df)

vm_cols = set(vendor_master_df.columns)
vendor_id_col    = pick_col(vm_cols, ['vendor_id', 'vendor_no', 'factory_id', 'supplier_id', 'id'])
vendor_short_col = pick_col(vm_cols, ['vendor_short_name', 'vendor_shortname', 'short_name', 'vendor_code', 'vendor'])
vendor_full_col  = pick_col(vm_cols, ['vendor_name', 'vendor_full_name', 'supplier_name', 'factory_name', 'name'])
vendor_loc_col   = pick_col(vm_cols, ['location', 'country', 'vendor_location', 'factory_location', 'factory_country'])

# Keyed by Vendor ID (FTY number) — the stable, truly unique identifier.
# vendorCode (generated string) is not stored in v3.
vendor_reference_map = {}
if vendor_id_col is not None:
    for _, vm_row in vendor_master_df.iterrows():
        vendor_id = none_if_nan(vm_row.get(vendor_id_col))
        if vendor_id is None:
            continue
        vendor_id    = str(vendor_id).strip()
        short_name   = none_if_nan(vm_row.get(vendor_short_col)) if vendor_short_col else None
        full_name    = none_if_nan(vm_row.get(vendor_full_col))  if vendor_full_col  else None
        loc_raw      = none_if_nan(vm_row.get(vendor_loc_col))   if vendor_loc_col   else None
        loc_name     = normalize_location(loc_raw) if loc_raw else None
        loc_code     = normalize_location_code(loc_name) if loc_name else None
        if loc_code:
            loc_name = LOCATION_CODE_TO_NAME.get(loc_code, loc_name)

        vendor_reference_map[vendor_id] = {
            'name':      full_name or short_name,
            'shortName': short_name,
            'location':  {'code': loc_code, 'name': loc_name} if loc_code else None,
        }
else:
    print("Warning: vendor_id column not found in Vendor Master — reference.vendors will be empty.")

# Reverse lookup: upper-cased short name → vendor ID.
# Used to resolve vendorId from the vendor_name column in the mold list.
vendor_name_upper_to_id = {
    info['shortName'].strip().upper(): vid
    for vid, info in vendor_reference_map.items()
    if info.get('shortName')
}

# Fill missing vendor locations from mold list rows.
# Vendor master may not carry a location column; mold rows always do.
# Business rule: one vendor = one location, so first occurrence is definitive.
for _, row in df.iterrows():
    v_name = none_if_nan(row.get('vendor_name'))
    if v_name is None:
        continue
    vid = vendor_name_upper_to_id.get(v_name.strip().upper())
    if vid is None or vendor_reference_map[vid].get('location') is not None:
        continue
    loc_name = none_if_nan(row.get('location'))  # already normalized by step 3a
    loc_code = normalize_location_code(loc_name) if loc_name else None
    if loc_code:
        loc_name = LOCATION_CODE_TO_NAME.get(loc_code, loc_name)
        vendor_reference_map[vid]['location'] = {'code': loc_code, 'name': loc_name}

print(f"vendor_reference_map: {len(vendor_reference_map)} vendors")

vendor_reference_map: 74 vendors


### 4b. `style_map_by_mold`  (brand-keyed, with soleTypes)

In [8]:
# Intermediate: { moldCode → { brand → { styleName → set(soleTypes) } } }
# Sets accumulate 'Outsole' and/or 'Midsole' per (family, brand, style) tuple.
# Handles the mismatch case: a style using different families for OS vs MS now
# appears in each family's list with the correct soleTypes subset.
_raw_style_map = {}

for _, style_row in mold_style_df.iterrows():
    brand      = none_if_nan(style_row.get('Brand'))
    style_name = none_if_nan(style_row.get('Style_Name'))
    os_code    = normalize_mold_code(style_row.get('Outsole_Name'))
    ms_code    = normalize_mold_code(style_row.get('Midsole_Name'))

    if brand is None or style_name is None:
        continue

    if os_code:
        _raw_style_map             .setdefault(os_code, {})             .setdefault(brand, {})             .setdefault(style_name, set())             .add('Outsole')

    if ms_code:
        _raw_style_map             .setdefault(ms_code, {})             .setdefault(brand, {})             .setdefault(style_name, set())             .add('Midsole')

# Convert sets to sorted lists for JSON serialisation
style_map_by_mold = {
    mold_code: {
        brand: [
            {'styleName': sn, 'soleTypes': sorted(st)}
            for sn, st in styles.items()
        ]
        for brand, styles in brands.items()
    }
    for mold_code, brands in _raw_style_map.items()
}

print(f"style_map_by_mold: {len(style_map_by_mold)} mold families")

style_map_by_mold: 103 mold families


### 4c. `allocation_map_by_mold`  (per-component sourcing rules)

In [9]:
# { moldCode → { componentCode → [{factoryNumber, vendorId}] } }
# Sole Type is dropped — it is redundant when keyed by component code.
allocation_map_by_mold = {}

for _, alloc_row in mold_vendor_allocation_df.iterrows():
    mold_code_key  = normalize_mold_code(alloc_row.get('Mold Code'))
    sole_part      = none_if_nan(alloc_row.get('Sole Part'))
    component_code = component_code_from_name(sole_part)
    if mold_code_key is None or component_code is None:
        continue

    rule = {
        'factoryNumber': none_if_nan(alloc_row.get('Factory Number')),
        'vendorId':      none_if_nan(alloc_row.get('Vendor ID')),
    }
    comp_map  = allocation_map_by_mold.setdefault(mold_code_key, {})
    rule_list = comp_map.setdefault(str(component_code), [])
    if rule not in rule_list:
        rule_list.append(rule)

print(f"allocation_map_by_mold: {len(allocation_map_by_mold)} mold families")

allocation_map_by_mold: 29 mold families


### 4d. `special_sizing_by_mold_component`

Sheet columns = shoe sizes; cell values = mold sizes.

In [10]:
special_df = replace_whitespace(special_mold_raw_df.copy())
special_size_cols = [c for c in special_df.columns if str(c).strip().startswith('Size ')]

special_sizing_by_mold_component = {}
for _, sp_row in special_df.iterrows():
    mold_code_key      = normalize_mold_code(sp_row.get('Mold Code'))
    component_code_key = component_code_from_name(sp_row.get('Sole Part'))
    if mold_code_key is None or component_code_key is None:
        continue

    key = (mold_code_key, str(component_code_key))
    mold_to_shoes = special_sizing_by_mold_component.setdefault(key, {})

    for col in special_size_cols:
        shoe_size     = str(col).replace('Size', '').strip()  # col header = shoe size
        mold_size_raw = normalize_numeric_or_text(sp_row.get(col))  # cell value = mold size
        if mold_size_raw is None:
            continue
        try:
            mold_size_f = float(str(mold_size_raw))
        except (ValueError, TypeError):
            continue
        mold_size_key = str(int(mold_size_f)) if mold_size_f.is_integer() else str(mold_size_f)

        shoe_list = mold_to_shoes.setdefault(mold_size_key, [])
        if shoe_size not in shoe_list:
            shoe_list.append(shoe_size)

### 4e. Sizing inference

In [11]:
STANDARD_MIN = 3.0
STANDARD_MAX = 13.0


def _size_str(v):
    v = round(v, 1)
    return str(int(v)) if v == int(v) else str(v)


def infer_coverage(base_sizes, overrides=None):
    """
    Infer shoe-size coverage for each base size in a single vendor entry.

    base_sizes : list[float]            – sizes where qty is recorded for this vendor
    overrides  : dict[str, list[str]]   – manual overrides {size_str: [shoe_sizes]}

    Returns {base_size_str: [shoe_size_str, ...]} if the sizing pattern is regular,
    or None if it cannot be classified (irregular gaps → quarantine).

    Step detection uses the standard range [STANDARD_MIN, STANDARD_MAX] only:
      uniform step s  →  n = round(s / 0.5) shoe sizes per mold
        step 0.5 → 1:1,  step 1.0 → 1:2,  step 1.5 → 1:3, …
      non-uniform     →  None (quarantine)
      0 or 1 standard sizes → assume step 0.5 (1:1)

    Extension sizes (outside standard range):
      fits  (offset from standard_min is a multiple of step) → same n coverage
      doesn't fit → 1:1 fallback
    """
    if overrides is None:
        overrides = {}

    non_overridden = [m for m in base_sizes if _size_str(m) not in overrides]
    standard       = sorted(m for m in non_overridden if STANDARD_MIN <= m <= STANDARD_MAX)

    if len(standard) >= 2:
        gaps = [round(standard[i + 1] - standard[i], 1) for i in range(len(standard) - 1)]
        if len(set(gaps)) != 1:
            return None
        step = gaps[0]
    else:
        step = 0.5

    n     = round(step / 0.5)
    s_min = standard[0] if standard else (min(non_overridden) if non_overridden else None)

    result = {}
    for m in base_sizes:
        key = _size_str(m)
        if key in overrides:
            result[key] = overrides[key]
            continue
        if s_min is not None:
            remainder = round((m - s_min) % step, 9)
            fits = remainder < 1e-9 or abs(remainder - step) < 1e-9
        else:
            fits = True
        count       = n if fits else 1
        result[key] = [_size_str(m + 0.5 * i) for i in range(count)]

    return result

## 5. Build and Export JSON Result

In [12]:
quarantine_records = []

result = {
    'schemaVersion': '5.0',
    'lastUpdated': date.today().isoformat(),
    'reference': {
        'vendors':       vendor_reference_map,
        'moldShops':     {},
        'moldTypes':     MOLD_TYPES,
        'positions':     {p: POSITION_NAMES[p] for p in ('PRI', 'BOT', 'HEEL')},
        'stages':        STAGES,
        'assetStatuses': ASSET_STATUSES,
        'moldSets':      MOLD_SETS,
    },
    'families': {},
}

for _, row in df.iterrows():
    mold_code = none_if_nan(row.get('mold_code')) or none_if_nan(row.get('mold_code_note'))
    if mold_code is None:
        continue
    family        = normalize_family(mold_code)
    mold_code_key = normalize_mold_code(mold_code)

    sole_type      = none_if_nan(row.get('part_name')) or 'Unknown'
    component_name = none_if_nan(row.get('component_name'))
    compound       = none_if_nan(row.get('compound'))
    component_code = component_code_from_name(component_name)
    if component_code is None:
        continue

    vendor_name    = none_if_nan(row.get('vendor_name'))
    mold_shop_name = none_if_nan(row.get('mold_shop'))
    mold_shop_code = normalize_code(mold_shop_name) if mold_shop_name else None

    vendor_id = vendor_name_upper_to_id.get(vendor_name.strip().upper()) if vendor_name else None

    if mold_shop_code and mold_shop_code not in result['reference']['moldShops']:
        result['reference']['moldShops'][mold_shop_code] = {'name': mold_shop_name}

    # ── Legacy → v3 Mold ID derivation ───────────────────────────────────────
    parsed = derive_component(component_code, component_name)
    if parsed is None or SOLE_TYPE_TO_TYPE.get(sole_type) != parsed[0]:
        quarantine_records.append({
            'reason':         'unmappable_component_code' if parsed is None else 'type_mismatch',
            'mold_code':      family,
            'sole_type':      sole_type,
            'component_code': str(component_code),
            'vendor_name':    vendor_name,
            'vendor_id':      vendor_id,
            'location':       none_if_nan(row.get('location')),
            'base_sizes':     [],
        })
        continue
    type_, position, stage = parsed
    mold_id = build_mold_id(family, type_, position, stage)

    # ── Family block ──────────────────────────────────────────────────────────
    fam = result['families'].setdefault(family, {
        'moldFamily':            family,
        'designFamily':          family,
        'isLegacyCode':          is_legacy_family_code(family),
        'notes':                 None,
        'stylesUsingThisFamily': {},
        'components':            {},
    })
    if not fam['stylesUsingThisFamily'] and mold_code_key in style_map_by_mold:
        fam['stylesUsingThisFamily'] = style_map_by_mold[mold_code_key]

    # ── Component block (Mold ID = product definition layer) ─────────────────
    comp = fam['components'].setdefault(mold_id, {
        'moldId':               mold_id,
        'legacyCode':           str(component_code),
        'type':                 type_,
        'position':             position,
        'stage':                stage,
        'soleType':             sole_type,
        'componentDescription': component_description(type_, position, stage),
        'constructionType':     None,  # derived in post-pass below
        'designCompound':       compound,
        'notes':                None,
        'sourcingRules':        [],
        'assets':               [],
    })
    if comp.get('designCompound') is None and compound is not None:
        comp['designCompound'] = compound

    if not comp['sourcingRules']:
        # Sourcing sheets still speak legacy codes ("Sole Part" = OS1 etc.)
        alloc_for_family = allocation_map_by_mold.get(mold_code_key, {})
        comp['sourcingRules'] = alloc_for_family.get(str(component_code), [])

    # ── Coverage inference (per vendor) ───────────────────────────────────────
    qty_by_base = {
        s: to_int_or_none(row.get(s))
        for s in size_cols if pd.notna(row.get(s))
    }
    base_floats = [float(s) for s in qty_by_base]

    sp_key    = (mold_code_key, str(component_code))
    overrides = special_sizing_by_mold_component.get(sp_key, {})

    inferred = infer_coverage(base_floats, overrides)
    if inferred is None:
        quarantine_records.append({
            'reason':         'irregular_sizing',
            'mold_code':      family,
            'sole_type':      sole_type,
            'component_code': str(component_code),
            'vendor_name':    vendor_name,
            'vendor_id':      vendor_id,
            'location':       none_if_nan(row.get('location')),
            'base_sizes':     sorted(qty_by_base.keys(), key=float),
        })
        continue  # family and component blocks already created above

    # ── Physical asset record (vendor-level asset group) ─────────────────────
    comment = none_if_nan(row.get('comments'))
    remark  = none_if_nan(row.get('remark'))
    if comment and remark and comment != remark:
        combined_comment = f"{comment} | {remark}"
    else:
        combined_comment = comment or remark

    coverage_lens = {len(inferred[b]) for b in qty_by_base}
    if not coverage_lens:
        size_grouping = None
    elif len(coverage_lens) == 1:
        size_grouping = f"1:{coverage_lens.pop()}"
    else:
        size_grouping = 'mixed'

    asset_entry = {
        'vendorId':         vendor_id,
        'moldShopCode':     mold_shop_code,
        'moldSet':          None,   # not in source data; A implied
        'revision':         None,   # not in source data; 0 implied
        'status':           None,   # Active/Inactive/In Repair/Retired — to be backfilled
        'ownership':        none_if_nan(row.get('mold_ownership')),
        'condition':        none_if_nan(row.get('mold_condition')),
        'conditionNote':    none_if_nan(row.get('mold_condition_note')),
        'moldCost':         to_float_or_none(row.get('a_mold_cost')),
        'initSeason':       none_if_nan(row.get('season')),
        'lastDemandSeason': none_if_nan(row.get('last_demand_season')),
        'capacity': {
            'dailyOutput':  to_float_or_none(row.get('daily_output')),
            'actualOutput': to_float_or_none(row.get('actual_output')),
        },
        'totalMoldQty':     to_int_or_none(row.get('total_mold_qty')),
        'sizeCoverage': [
            {'shoeSizes': inferred[b], 'qty': qty}
            for b, qty in sorted(qty_by_base.items(), key=lambda x: float(x[0]))
        ],
        'sizeGrouping':     size_grouping,
        'comments':         combined_comment,
    }
    if vendor_id is None and vendor_name:
        asset_entry['vendorName'] = vendor_name

    comp['assets'].append(asset_entry)

# ── Post-pass: derive Construction Type per (family, type) ───────────────────
# Positions are collected regardless of stage: a blocker is a production
# stage of the same position, not an extra piece in the assembled sole.
for fam in result['families'].values():
    positions_by_type = {}
    for comp in fam['components'].values():
        positions_by_type.setdefault(comp['type'], set()).add(comp['position'])
    for comp in fam['components'].values():
        comp['constructionType'] = derive_construction_type(positions_by_type[comp['type']])

with open('../data/mold_data.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2, ensure_ascii=False, default=str)

print(f'Exported {len(result["families"])} mold families to mold_data.json  '
      f'(schemaVersion {result["schemaVersion"]})')

quarantine_df = pd.DataFrame(quarantine_records)
if len(quarantine_df):
    print(f'\nQuarantined {len(quarantine_df)} mold entries:')
    print(quarantine_df['reason'].value_counts().to_string())
    display(quarantine_df)
else:
    print('\nNo quarantined entries.')

Exported 242 mold families to mold_data.json  (schemaVersion 5.0)



Quarantined 47 mold entries:
reason
irregular_sizing    47


,reason,mold_code,sole_type,component_code,vendor_name,vendor_id,location,base_sizes
0,irregular_sizing,SA-2551,Outsole,OS1,JIATAI-CN(HENGMAO),FTY000575,China,"[3, 5.5, 10.5, 12.5]"
1,irregular_sizing,SA-2654,Outsole,OS1,STELLA-ID(SABINA),FTY000720,Indonesia,"[3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5, 11.5..."
2,irregular_sizing,SA-2654-4E,Outsole,OS1,STELLA-ID(SABINA),FTY000720,Indonesia,"[3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5, 11.5..."
3,irregular_sizing,SA-1529,Outsole,OS1,FINECHEMICAL-CN,FTY000520,China,"[3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 8.5, ..."
4,irregular_sizing,SA-1638,Midsole,MS1,FINECHEMICAL-CN,FTY000520,China,"[3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 8.5, ..."
5,irregular_sizing,SA-2258,Midsole,MS1,GIAJIU-VN(FORTUNEART/JOYFUL),FTY000484,Vietnam,"[3, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5, 1..."
6,irregular_sizing,SA-2258,Midsole,MS2,GIAJIU-VN(FORTUNEART/JOYFUL),FTY000484,Vietnam,"[3, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5, 1..."
7,irregular_sizing,SA-2267,Midsole,MS1,GUOSHENG-CN,FTY000480,China,"[3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10, 10.5, ..."
8,irregular_sizing,SA-2267,Midsole,MS2,GUOSHENG-CN,FTY000480,China,"[3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10, 10.5, ..."
9,irregular_sizing,SA-2501,Midsole,MS2-1,GUOSHENG-CN,FTY000480,China,"[3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10, 10.5, ..."
